In [1]:
import json
import os
import time
from multiprocessing import Pool, cpu_count

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from ipydatagrid import DataGrid, TextRenderer
from IPython.display import HTML, display



def get_perf(ISIN, dateFrom, dateTo, type_code="DecalogCodePtf"):

    url = f"https://perfo-api.intramundi.com/performance/api/v1/valuationData/flatten?typeCode={type_code}&assetcode={ISIN}&dateFrom={dateFrom}&dateTo={dateTo}"
    # json_data = requests.get(url)

    # data=pd.read_json(json_data.text)

    data = pd.read_json(url)
    data["valuationDate"] = pd.to_datetime(
        data["valuationDate"], errors="coerce", utc=True
    )
    data["valuationDate"] = data["valuationDate"].apply(
        lambda x: x.replace(tzinfo=None)
    )
    data = data.set_index("valuationDate")
    data.index = data.index.round(freq="d")
    # data.index=data.index.normalize()

    return data

def safe_call(tasks):
    func, ticker, datestart, dateto, type_code = tasks

    try:
        return ticker, func(ticker, datestart, dateto, type_code)
    except Exception as e:
        return {ticker: str(e), "args": tasks}

def display_scrollable_df(df, max_height="50vh", max_width="90vw"):
    style = f"""
    <div style="
        display: flex;
        justify-content: center;
        padding: 20px;
    ">
        <div style="
            overflow: auto;
            max-height: {max_height};
            max-width: {max_width};
            width: 100%;
            border: 1px solid #444;
            padding: 10px;
            background-color: #000;
            color: #eee;
            font-family: 'Arial Narrow', Arial, sans-serif;
            box-sizing: border-box;
        ">
            {df.to_html(classes='table', border=0, index=True)}
        </div>
    </div>
    """
    return HTML(style)

def display_app(ISINs):
    date_start = widgets.Text(
        step=1,
        description="Start (YYYYMMDD)",
        disabled=False,
        display="flex",
        flex_flow="column",
        align_items="stretch",style={'description_width':'auto'}
    )

    date_end = widgets.Text(
        step=1,
        description="End (YYYYMMDD)",
        disabled=False,
        display="flex",
        flex_flow="column",
        align_items="stretch",style={'description_width':'auto'}
    )

    code_type = widgets.Dropdown(
        options=["DecalogCodePtf", "ISINCodePtf"],
        value="ISINCodePtf",
        description="Code Type",
        disabled=False,
        display="flex",
        flex_flow="column",
        align_items="stretch",style={'description_width':'auto'}
    )

    def get_data_on_click(b):

        global perf_dict, not_found
        tasks = [
            (get_perf, t, date_start.value, date_end.value, code_type.value)
            for t in ISINs
        ]
        perf_dict = {}
        start = time.time()
        not_found = []

        with Pool(processes=min(5, cpu_count() * 2)) as pool:

            for ticker, data in pool.imap_unordered(safe_call, tasks):

                if type(data) is not str:

                    if "benchPerf" in data.columns:
                        data["Excess Returns"] = (
                            data["grossperf"] - data["benchPerf"]
                        ) / 100
                        data = data.rename(
                            columns={
                                "grossBase100": "NAV Base 100",
                                "benchmarkValue": "Benchmark Base 100",
                                "grossperf": "Share Class Return",
                                "benchPerf": "Benchmark Return",
                            }
                        )

                    else:
                        data["benchPerf"] = data["grossperf"].fillna(0) / 100
                        data["Excess Returns"] = (data["grossperf"]) / 100
                        data = data.rename(
                            columns={
                                "grossBase100": "NAV Base 100",
                                "benchmarkValue": "Benchmark Base 100",
                                "grossperf": "Share Class Return",
                                "benchPerf": "Benchmark Return",
                            }
                        )

                    perf_dict[ticker] = data
                else:
                    not_found.append(ticker)

            # results=dict(pool.map(safe_call,tasks))
            # results=pool.map(safe_call,tasks)
        #     print(results)

        #     for r in results:
        #         print(r)

        with data_output:
            data_output.clear_output()
            display(
                display_scrollable_df(pd.DataFrame(not_found, columns=["Missing ETFs"]))
            )
            display(
                display_scrollable_df(
                    pd.DataFrame(list(perf_dict.keys()), columns=["Retrieved ETFs"])
                )
            )

        final = time.time()
        print(f"{final-start:.2f}")

    def get_atypical_perf(b):

        global flagged_cumulative, flagged_daily

        flagged_cumulative = {}
        flagged_daily = {}

        # Daily Control and Cumulative Control on Excess Returns to flag atypical Performance#

        te = {}
        vol = {}
        for key in perf_dict:
            temp = (perf_dict[key]["Excess Returns"]).resample("ME").std() * np.sqrt(
                252
            )
            temp_vol = (perf_dict[key]["Share Class Return"]).resample(
                "ME"
            ).std() * np.sqrt(252)
            te[key] = temp
            vol[key] = temp_vol

        for ISIN in perf_dict:

            temp = perf_dict[ISIN]
            index = np.where(abs(temp["Excess Returns"]) > daily_limit.value / 10000)
            flagged_daily[ISIN] = temp.iloc[index]

            if (
                abs((1 + temp["Excess Returns"]).cumprod().iloc[-1] - 1)
                > cumulative_limit.value / 10000
            ):
                flagged_cumulative[ISIN] = (1 + temp["Excess Returns"]).cumprod()

            else:

                continue

        # Compute the cumulative Excess Returns to be used as a plot
        excess_returns_cumulative = {}
        for key in perf_dict:
            excess_returns_cumulative[key] = (
                1 + perf_dict[key]["Excess Returns"]
            ).cumprod()

        # excess_returns_cumulative_dataframe=pd.DataFrame(excess_returns_cumulative)

        # Get last value of Cumulative Excess Return (to see if above limit and be inserted in a table)

        cumulative = {}
        for ISIN in perf_dict:
            temp = perf_dict[ISIN]
            cumulative[ISIN] = (
                (1 + temp["Excess Returns"]).cumprod().iloc[-1] - 1
            ) * 10000

        # Summary Table for the daily returns and ETFs flagged#

        summary_daily = {}

        for key in flagged_daily:

            try:
                temp = flagged_daily[key]
                count = temp.shape[0]
                max_dev = (temp["Excess Returns"].max() * 10000).round(4)
                min_dev = (temp["Excess Returns"].min() * 10000).round(4)
                date_max = temp["Excess Returns"].idxmax()
                date_min = temp["Excess Returns"].idxmin()

                summary_daily[key] = [count, max_dev, date_max, min_dev, date_min]

            except Exception as e:
                print(f"Data not found for {key}")

                pass

        excess = {}
        benchmark = {}
        share_class_returns = {}

        for key in perf_dict:

            temp = perf_dict[key]
            wo_dup = temp[~temp.index.duplicated()]
            excess[key] = wo_dup["Excess Returns"]

            benchmark[key] = wo_dup["Benchmark Return"]

            share_class_returns[key] = wo_dup["Share Class Return"]

        global daily_deviation

        daily_deviation = pd.DataFrame(
            summary_daily,
            index=[
                "Numbers of Violations",
                "Max Upside Deviation in BPS",
                "Date of Max (Upside) Deviation",
                "Max Downside Deviation in BPS",
                "Date of Max (Downside) Deviation",
            ],
        ).T

        global returns_dataframe, benchmark_returns, excess_returns_dataframe, cumulative_dataframe, monthly_te, monthly_vol

        returns_dataframe = pd.DataFrame(share_class_returns).sort_index()
        benchmark_returns = pd.DataFrame(benchmark).sort_index()
        excess_returns_dataframe = pd.DataFrame(excess).sort_index()

        monthly_te = pd.DataFrame(te).sort_index()
        monthly_vol = pd.DataFrame(vol).sort_index()

        cumulative_dataframe = pd.DataFrame(
            cumulative.values(),
            index=cumulative.keys(),
            columns=["Final Excess Return (Bps)"],
        )
        cumulative_dataframe = cumulative_dataframe.sort_values(
            by="Final Excess Return (Bps)", ascending=False
        )

        fund_list = list(perf_dict.keys())
        
        global selected_fund
        selected_fund=widgets.Dropdown(
        options=fund_list,
        disabled=False,
    )


        def get_excel(b):

            with pd.ExcelWriter(
                "Atypical Performance.xlsx", engine="openpyxl"
            ) as writer:

                returns_dataframe.to_excel(writer, sheet_name="Returns", index=True)
                benchmark_returns.to_excel(writer, sheet_name="Benchmark", index=True)
                excess_returns_dataframe.to_excel(
                    writer, sheet_name="Excess Returns", index=True
                )
                daily_deviation.to_excel(
                    writer, sheet_name="Daily Violations", index=True
                )
                cumulative_dataframe.to_excel(
                    writer, sheet_name="Cumulative Violations", index=True
                )
                monthly_te.to_excel(
                    writer, sheet_name="Monthly Tracking Error in %", index=True
                )
                monthly_vol.to_excel(
                    writer, sheet_name="Monthly Volatility in %", index=True
                )
            
            print("File Generated")
                
        bt_excel = widgets.Button(
            description="Get Excel",
            layout=widgets.Layout(
                display="flex",
                justify_content="center",
                align_items="center",
                spacing="10px",
                width="auto",
            ),
        )

        bt_excel.on_click(get_excel)

        with button_atypical_table:
            button_atypical_table.clear_output()
            display(display_scrollable_df(daily_deviation))
            display(display_scrollable_df(cumulative_dataframe))
            display(bt_excel)
    def get_time_series(value1):

        fund=returns_dataframe[value1]
        bench=benchmark_returns[value1]
        excess=excess_returns_dataframe[value1]

        returns_series=pd.concat([fund,bench,excess],axis=1)
        returns_series.columns=['Fund','Benchmark','Excess Return']

        returns_series=(1+returns_series/100).cumprod()*100

        return returns_series


    def get_monthly_tracking_error(value1):

        excess=excess_returns_dataframe[value1]
        excess.columns=['Monthly Tracking Error in BPS']
        monthly_tracking_error=excess.resample('ME').std()*np.sqrt(252)*100

        return monthly_tracking_error

    def get_monthly_vol(value1):

        returns=returns_dataframe[value1].dropna()
        returns.columns=['Monthly Vol in %']
        monthly_volatility=returns.resample('ME').std()*np.sqrt(252)

        return monthly_volatility

    def plot_chart(b):
        
        
        time_series=get_time_series(selected_fund.value).dropna()
        monthly_te=get_monthly_tracking_error(selected_fund.value).dropna()
        monthly_vol=get_monthly_vol(selected_fund.value).dropna()
        
        plt.style.use("dark_background")
        
        fig=plt.figure()
        plt.plot(time_series[['Fund','Benchmark']])
        plt.title("Cumulative Returns")
        plt.xticks(rotation=45)
        plt.xlabel("Date")
        plt.tight_layout()
        
        fig2=plt.figure()
        plt.plot(time_series['Excess Return'])
        plt.title("Cumulative Excess Returns")
        plt.xticks(rotation=45)
        plt.xlabel("Date")
        plt.tight_layout()
        
        fig3=plt.figure()
        plt.plot(monthly_te)
        plt.title("Monthly Tracking Error in BPS")
        plt.xticks(rotation=45)
        plt.xlabel("Date")
        plt.tight_layout()

        
        
        fig4=plt.figure()
        plt.plot(monthly_vol)
        plt.title("Monthly Vol in %")
        plt.xticks(rotation=45)
        plt.xlabel("Date")
        plt.tight_layout()

        with chart_output:
            chart_output.clear_output()
            plt.show()
            display(display_scrollable_df(time_series))
            
    button_data = widgets.Button(description="Get Data")
    data_output = widgets.Output()
    button_data.on_click(get_data_on_click)

    button_atypical = widgets.Button(description="Get Results")
    button_atypical_table = widgets.Output()
    button_atypical.on_click(get_atypical_perf)
    
    selected_fund=widgets.Dropdown(
        options=ISINs,
        disabled=False,
    )
    

    button_chart = widgets.Button(description="Get Chart")
    chart_output = widgets.Output()
    button_chart.on_click(plot_chart)
    
    daily_limit = widgets.BoundedFloatText(
        value=5, step=0.1, description="Daily Limit (BPS)", disabled=False,style={'description_width':'auto'}
    )

    cumulative_limit = widgets.BoundedFloatText(
        value=20, step=0.1, description="Cumulative Limit (BPS)", disabled=False,style={'description_width':'auto'}
    )

    parameters_ui = widgets.VBox(
        [date_start, date_end, code_type, button_data, data_output],
        layout=widgets.Layout(
            display="flex",
            justify_content="center",
            align_items="center",
            spacing="auto",
            width="auto",
        ),
    )

    limit_ui = widgets.VBox([daily_limit, cumulative_limit, button_atypical, button_atypical_table],
                           layout=widgets.Layout(
            display="flex",
            justify_content="center",
            align_items="center",
            spacing="auto",
            width="auto"))

    app = widgets.VBox([parameters_ui, limit_ui])
    chart=widgets.VBox([selected_fund,button_chart,chart_output])
    tab_contents = ["Control","Chart"]
    
    children = [app,chart]
    tab = widgets.Tab()
    tab.children = children
    for i, title in enumerate(tab_contents):
        tab.set_title(i, title)
    display(tab)

In [2]:
ETF = pd.read_excel("ETF.xlsx", index_col=0)
# ETF = pd.read_excel("Scope.xlsx")
# ETF = ETF.set_index(ETF.columns[0])
ISINs = list(ETF.index)
scope=pd.read_excel('Scope.xlsx',index_col=0)
scope_funds=list(scope.index)

In [3]:
display_app(scope_funds)